In [11]:
!pip install scikit-learn
!pip install matplotlib
!pip install seaborn
!pip install scipy

import pandas as pd
import numpy as np
from sklearn import preprocessing
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import re
import string
print("install is done")


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


install is done


In [12]:

df = pd.read_csv('data/1662574418893344.csv')
df.head()

,Food_ID,Name,C_Type,Veg_Non,Describe
0,1,summer squash salad,Healthy Food,veg,"white balsamic vinegar, lemon juice, lemon rin..."
1,2,chicken minced salad,Healthy Food,non-veg,"olive oil, chicken mince, garlic (minced), oni..."
2,3,sweet chilli almonds,Snack,veg,"almonds whole, egg white, curry leaves, salt, ..."
3,4,tricolour salad,Healthy Food,veg,"vinegar, honey/sugar, soy sauce, salt, garlic ..."
4,5,christmas cake,Dessert,veg,"christmas dry fruits (pre-soaked), orange zest..."


In [13]:
def remove_punctuation(s):
    return s.translate(str.maketrans('', '', string.punctuation))
df['Describe'] = df['Describe'].apply(remove_punctuation)
df.head()

,Food_ID,Name,C_Type,Veg_Non,Describe
0,1,summer squash salad,Healthy Food,veg,white balsamic vinegar lemon juice lemon rind ...
1,2,chicken minced salad,Healthy Food,non-veg,olive oil chicken mince garlic minced onion sa...
2,3,sweet chilli almonds,Snack,veg,almonds whole egg white curry leaves salt suga...
3,4,tricolour salad,Healthy Food,veg,vinegar honeysugar soy sauce salt garlic clove...
4,5,christmas cake,Dessert,veg,christmas dry fruits presoaked orange zest lem...


In [14]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['Describe'])
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
indices = pd.Series(df.index, index=df['Name']).drop_duplicates()

def get_recommendations(title, cosine_sim=cosine_sim):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:6]
    food_indices = [i[0] for i in sim_scores]
    return df['Name'].iloc[food_indices]

features = ['C_Type','Veg_Non', 'Describe']
def create_soup(x):
    return x['C_Type'] + " " + x['Veg_Non'] + " " + x['Describe']
df['soup'] = df.apply(create_soup, axis=1)
df.head()


,Food_ID,Name,C_Type,Veg_Non,Describe,soup
0,1,summer squash salad,Healthy Food,veg,white balsamic vinegar lemon juice lemon rind ...,Healthy Food veg white balsamic vinegar lemon ...
1,2,chicken minced salad,Healthy Food,non-veg,olive oil chicken mince garlic minced onion sa...,Healthy Food non-veg olive oil chicken mince g...
2,3,sweet chilli almonds,Snack,veg,almonds whole egg white curry leaves salt suga...,Snack veg almonds whole egg white curry leaves...
3,4,tricolour salad,Healthy Food,veg,vinegar honeysugar soy sauce salt garlic clove...,Healthy Food veg vinegar honeysugar soy sauce ...
4,5,christmas cake,Dessert,veg,christmas dry fruits presoaked orange zest lem...,Dessert veg christmas dry fruits presoaked ora...


In [15]:
count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(df['soup'])
cosine_sim2 = cosine_similarity(count_matrix, count_matrix)
df = df.reset_index()
indices = pd.Series(df.index, index=df['Name'])
get_recommendations('tricolour salad')


103             chilli chicken
1         chicken minced salad
27     vegetable som tam salad
282          veg hakka noodles
166             veg fried rice
Name: Name, dtype: object

In [16]:
rating = pd.read_csv('data/ratings.csv')
rating.head()

,User_ID,Food_ID,Rating
0,1.0,88.0,4.0
1,1.0,46.0,3.0
2,1.0,24.0,5.0
3,1.0,25.0,4.0
4,2.0,49.0,1.0


In [17]:
rating = rating[:511]
food_rating = rating.groupby(by = 'Food_ID').count()
food_rating = food_rating['Rating'].reset_index().rename(columns={'Rating':'Rating_count'})
user_rating = rating.groupby(by='User_ID').count()
user_rating = user_rating['Rating'].reset_index().rename(columns={'Rating':'Rating_count'})
rating_matrix = rating.pivot_table(index='Food_ID',columns='User_ID',values='Rating').fillna(0)
rating_matrix.head()



User_ID,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,...,91.0,92.0,93.0,94.0,95.0,96.0,97.0,98.0,99.0,100.0
Food_ID,,,,,,,,,,,,,,,,,,,,,
1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,0.0,0.0,0.0,7.0,0.0,0.0


In [18]:
csr_rating_matrix =  csr_matrix(rating_matrix.values)
recommender = NearestNeighbors(metric='cosine')
recommender.fit(csr_rating_matrix)

NearestNeighbors(metric='cosine')

In [19]:
def Get_Recommendations(title):
    user= df[df['Name']==title]
    user_index = np.where(rating_matrix.index == int(user['Food_ID'].iloc[0]))[0][0]
    user_ratings = rating_matrix.iloc[user_index]

    reshaped = user_ratings.values.reshape(1,-1)
    distances, indices = recommender.kneighbors(reshaped,n_neighbors=16)
    
    nearest_neighbors_indices = rating_matrix.iloc[indices[0]].index[1:]
    nearest_neighbors = pd.DataFrame({'Food_ID': nearest_neighbors_indices})
    
    result = pd.merge(nearest_neighbors,df,on='Food_ID',how='left')

    return result.head()

Get_Recommendations('chicken minced salad')


,Food_ID,index,Name,C_Type,Veg_Non,Describe,soup
0,30.0,29,couscous with ratatouille - tangy tomato sauce,French,veg,for the cous cous plain couscous extra virgin ...,French veg for the cous cous plain couscous ex...
1,195.0,194,egg in a blanket,French,non-veg,eggs brown bread slices butter chilli flakes o...,French non-veg eggs brown bread slices butter ...
2,256.0,255,apple kheer,Dessert,veg,apples basmati rice nuscovado sugar you can al...,Dessert veg apples basmati rice nuscovado suga...
3,175.0,174,chicken paella,Mexican,non-veg,chicken oil salt and pepper paprika powder chi...,Mexican non-veg chicken oil salt and pepper pa...
4,86.0,85,roast turkey with cranberry sauce,Healthy Food,non-veg,whole turkey butter onion celery crumbled sage...,Healthy Food non-veg whole turkey butter onion...


In [20]:
from collections import Counter
def Get_General_Recommendations(titles, top_n=5, n_neighbors=16):
    all_recommended_ids = [] 
    
    for title in titles:

        user = df[df['Name'] == title]
        if user.empty:
            print(f"Title '{title}' not found in the dataset. Skipping.")
            continue
        
        user_id = int(user['Food_ID'].iloc[0])
        user_index_array = np.where(rating_matrix.index == user_id)[0]
        
        if len(user_index_array) == 0:
            print(f"Food_ID {user_id} for title '{title}' not found in rating_matrix. Skipping.")
            continue
        
        user_index = user_index_array[0]
        user_ratings = rating_matrix.iloc[user_index]
        
        reshaped = user_ratings.values.reshape(1, -1)
        
        distances, indices = recommender.kneighbors(reshaped, n_neighbors=n_neighbors)
        
        recommended_ids = rating_matrix.iloc[indices[0]].index[1:]
        all_recommended_ids.extend(recommended_ids)
    
    if not all_recommended_ids:
        print("No recommendations found for the provided titles.")
        return pd.DataFrame()
    
    rec_counter = Counter(all_recommended_ids)
    
    sorted_rec_ids = [food_id for food_id, count in rec_counter.most_common()]
    
    recommendations_df = df[df['Food_ID'].isin(sorted_rec_ids)].copy()
    
    recommendations_df['Frequency'] = recommendations_df['Food_ID'].apply(lambda x: rec_counter[x])
    
    recommendations_df = recommendations_df.sort_values(by='Frequency', ascending=False)
    
    return recommendations_df.head(top_n)

titles_list = ['tricolour salad', 'sugar free modak', 'baked shankarpali ']
final_recommendations = Get_General_Recommendations(titles_list, top_n=10)
final_recommendations.head(10)

,index,Food_ID,Name,C_Type,Veg_Non,Describe,soup,Frequency
300,300,301,brown rice,Healthy Food,veg,riety of rice,Healthy Food veg riety of rice,2
302,302,303,red rice,Healthy Food,veg,riety of rice,Healthy Food veg riety of rice,2
298,298,299,kolim / jawla,Indian,veg,dried fish named kolim or jawla found in coast...,Indian veg dried fish named kolim or jawla fou...,2
299,299,300,black rice,Healthy Food,veg,riety of rice,Healthy Food veg riety of rice,2
303,303,304,rice,Indian,veg,boiled rice,Indian veg boiled rice,2
125,125,126,andhra crab meat masala,Indian,non-veg,processed crab meat refined oil curry leaves g...,Indian non-veg processed crab meat refined oil...,2
301,301,302,koldil chicken,Chinese,non-veg,made with banana flower an assamese specialty,Chinese non-veg made with banana flower an ass...,2
48,48,49,christmas tree pizza,Italian,veg,pizza dough 2 boules red pepper red onion basi...,Italian veg pizza dough 2 boules red pepper re...,1
50,50,51,christmas chocolate fudge cookies,Dessert,veg,unsalted butter brown sugar chocolate chocolat...,Dessert veg unsalted butter brown sugar chocol...,1
24,24,25,cashew nut cookies,Dessert,veg,cashew paste ghee khaand a sweetening agent an...,Dessert veg cashew paste ghee khaand a sweeten...,1


In [21]:
from flask import Flask, request, jsonify
app = Flask(__name__)

# Define recommendation function
def Get_Recommendations(title):
    user = df[df['Name'] == title]

    if user.empty:
        return None  # If food not found, return None

    user_index = np.where(rating_matrix.index == int(user['Food_ID'].iloc[0]))[0][0]
    user_ratings = rating_matrix.iloc[user_index]

    reshaped = user_ratings.values.reshape(1, -1)
    distances, indices = recommender.kneighbors(reshaped, n_neighbors=16)

    nearest_neighbors_indices = rating_matrix.iloc[indices[0]].index[1:]
    nearest_neighbors = pd.DataFrame({'Food_ID': nearest_neighbors_indices})

    result = pd.merge(nearest_neighbors, df, on='Food_ID', how='left')

    return result[['Food_ID', 'Name', 'Category']].head().to_dict(orient='records')

# Flask API endpoint
@app.route('/recommend', methods=['GET'])
def recommend():
    title = request.args.get('title')  # Get food title from request

    if not title:
        return jsonify({'error': 'Title parameter is required'}), 400

    recommendations = Get_Recommendations(title)

    if recommendations is None:
        return jsonify({'error': 'Food item not found'}), 404

    return jsonify(recommendations)

# Run Flask app
if __name__ == '__main__':
    app.run(debug=True,port= 5500)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5500
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

C:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
